# MongoDB Logical Export, Separate Restore, and Verification

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST4714_DB_admin/blob/main/CST4714_OER_Rebuild/notebooks/05_mongodb_logical_recovery.ipynb)

This notebook creates a collection-level canonical Extended JSON artifact, restores
it under a different database name, and verifies identifiers, counts, BSON date
types, a meaningful query, indexes, and validation behavior.

The exercise is intentionally transparent. It is **not** mislabeled as a complete
Atlas backup. Atlas Free does not provide native backups; `mongodump` and
`mongorestore` are the documented database-level tools.

In [1]:
%pip -q install pymongo mongomock

Note: you may need to restart the kernel to use updated packages.


In [2]:
from datetime import datetime, timezone
from getpass import getpass
import hashlib
from pathlib import Path

import mongomock
from bson import json_util
from bson.json_util import CANONICAL_JSON_OPTIONS
from pymongo import ASCENDING, DESCENDING, MongoClient
from pymongo.errors import OperationFailure
from pymongo.server_api import ServerApi

USE_ATLAS = False  # Change only in the in-class Atlas lab.
DATABASE_SUFFIX = "offline"  # Use a unique suffix before enabling Atlas.

if USE_ATLAS:
    if DATABASE_SUFFIX == "offline":
        raise ValueError("Replace DATABASE_SUFFIX with a unique course value before using Atlas.")
    mongodb_uri = getpass("Paste the temporary Atlas URI: ")
    client = MongoClient(
        mongodb_uri,
        server_api=ServerApi("1", strict=True, deprecation_errors=True),
        serverSelectionTimeoutMS=10000,
        timeoutMS=10000,
    )
    print("Atlas ping:", client.admin.command("ping"))
else:
    client = mongomock.MongoClient()
    print("Using the offline in-memory MongoDB-compatible path.")

SOURCE_DB = f"cst4714_recovery_source_{DATABASE_SUFFIX}"
RESTORE_DB = f"cst4714_recovery_restore_{DATABASE_SUFFIX}"
EXPORT_FILE = Path("/tmp/cst4714_tickets_canonical_extjson.json")
print("Source:", SOURCE_DB)
print("Restore target:", RESTORE_DB)

Using the offline in-memory MongoDB-compatible path.
Source: cst4714_recovery_source_offline
Restore target: cst4714_recovery_restore_offline


## 1. Create the Source Collection and Operational Rules

The source collection has a focused validator in Atlas and a compound index. The
offline library supports the documents and index but not server-side validation,
so the notebook labels that limitation rather than pretending the feature ran.

In [3]:
client.drop_database(SOURCE_DB)
client.drop_database(RESTORE_DB)
source_database = client[SOURCE_DB]

ticket_validator = {
    "$jsonSchema": {
        "bsonType": "object",
        "required": ["ticket_id", "status", "priority", "subject", "opened_at"],
        "properties": {
            "ticket_id": {"bsonType": ["int", "long"]},
            "status": {"enum": ["new", "open", "in_progress", "resolved", "closed"]},
            "priority": {"enum": ["low", "medium", "high", "urgent"]},
            "subject": {"bsonType": "string"},
            "opened_at": {"bsonType": "date"},
        },
    }
}

if USE_ATLAS:
    source_database.create_collection("tickets", validator=ticket_validator)
else:
    source_database.create_collection("tickets")
    print("Offline path: server-side $jsonSchema validation is not implemented by mongomock.")

source_tickets = source_database["tickets"]
source_tickets.create_index([("status", ASCENDING), ("opened_at", DESCENDING)])

source_documents = [
    {"ticket_id": 1001, "status": "open", "priority": "high",
     "subject": "Streetlight dark near bus stop",
     "opened_at": datetime(2026, 2, 1, 23, 10, tzinfo=timezone.utc)},
    {"ticket_id": 1002, "status": "in_progress", "priority": "medium",
     "subject": "Missed recycling pickup",
     "opened_at": datetime(2026, 2, 2, 15, 45, tzinfo=timezone.utc)},
    {"ticket_id": 1003, "status": "resolved", "priority": "urgent",
     "subject": "Low water pressure",
     "opened_at": datetime(2026, 2, 3, 12, 5, tzinfo=timezone.utc)},
    {"ticket_id": 1004, "status": "new", "priority": "low",
     "subject": "Broken bench slat",
     "opened_at": datetime(2026, 2, 4, 17, 20, tzinfo=timezone.utc)},
    {"ticket_id": 1005, "status": "resolved", "priority": "high",
     "subject": "Overflowing corner bin",
     "opened_at": datetime(2026, 2, 5, 14, 0, tzinfo=timezone.utc)},
]
source_tickets.insert_many(source_documents)

print("Source count:", source_tickets.count_documents({}))
print("Source indexes:", sorted(index["name"] for index in source_tickets.list_indexes()))

Offline path: server-side $jsonSchema validation is not implemented by mongomock.
Source count: 5
Source indexes: ['_id_', 'status_1_opened_at_-1']


## 2. Create and Inspect Canonical Extended JSON

Canonical Extended JSON preserves BSON type information such as dates and ObjectId
values in a JSON-compatible representation. File size and SHA-256 help identify
the exact artifact; they do not prove it can be restored.

In [4]:
documents_to_export = list(source_tickets.find({}).sort("ticket_id", ASCENDING))
export_text = json_util.dumps(
    documents_to_export,
    json_options=CANONICAL_JSON_OPTIONS,
    indent=2,
)
EXPORT_FILE.write_text(export_text + "\n", encoding="utf-8")

export_bytes = EXPORT_FILE.read_bytes()
export_sha256 = hashlib.sha256(export_bytes).hexdigest()
print("Artifact:", EXPORT_FILE)
print("Bytes:", len(export_bytes))
print("SHA-256:", export_sha256)
print("First 300 characters:\n", export_text[:300])

Artifact: /tmp/cst4714_tickets_canonical_extjson.json
Bytes: 1515
SHA-256: 919bb2f7cf1b47595a107da22114d676f858f089fb1318b0cc9d7aa27dadb0b2
First 300 characters:
 [
  {
    "ticket_id": {
      "$numberInt": "1001"
    },
    "status": "open",
    "priority": "high",
    "subject": "Streetlight dark near bus stop",
    "opened_at": {
      "$date": {
        "$numberLong": "1769987400000"
      }
    },
    "_id": {
      "$oid": "6a5485dfb840015d65c3c488"
  


## 3. Restore Into a Different Database

The restore database name is different from the source. Parsing with `json_util`
reconstructs BSON-aware Python values before insertion.

In [5]:
restore_database = client[RESTORE_DB]
restore_tickets = restore_database["tickets"]

restored_documents = json_util.loads(EXPORT_FILE.read_text(encoding="utf-8"))
restore_result = restore_tickets.insert_many(restored_documents)
print("Restored documents:", len(restore_result.inserted_ids))
print("Restore count:", restore_tickets.count_documents({}))

Restored documents: 5
Restore count: 5


## 4. Verify Data, Identity, Type, and Behavior

Counts are only one check. We compare ticket identifiers, inspect the restored
date type, and run the active-ticket question.

In [6]:
source_ids = [doc["ticket_id"] for doc in source_tickets.find({}, {"_id": 0, "ticket_id": 1}).sort("ticket_id", 1)]
restore_ids = [doc["ticket_id"] for doc in restore_tickets.find({}, {"_id": 0, "ticket_id": 1}).sort("ticket_id", 1)]
print("Source IDs:", source_ids)
print("Restore IDs:", restore_ids)
assert source_ids == restore_ids

restored_sample = restore_tickets.find_one({"ticket_id": 1001})
print("Restored opened_at type:", type(restored_sample["opened_at"]).__name__)
assert isinstance(restored_sample["opened_at"], datetime)

active = list(
    restore_tickets.find(
        {"status": {"$in": ["new", "open", "in_progress"]}},
        {"_id": 0, "ticket_id": 1, "status": 1},
    ).sort("ticket_id", 1)
)
print("Active restored tickets:", active)

Source IDs: [1001, 1002, 1003, 1004, 1005]
Restore IDs: [1001, 1002, 1003, 1004, 1005]
Restored opened_at type: datetime
Active restored tickets: [{'ticket_id': 1001, 'status': 'open'}, {'ticket_id': 1002, 'status': 'in_progress'}, {'ticket_id': 1004, 'status': 'new'}]


## 5. Identify What the JSON Artifact Omitted

Collection indexes and validators are database metadata. The document-only export
did not recreate them automatically.

In [7]:
print("Restore indexes before repair:", sorted(index["name"] for index in restore_tickets.list_indexes()))
print("Expected: only the automatic _id_ index before manual recreation.")

restore_tickets.create_index([("status", ASCENDING), ("opened_at", DESCENDING)])

if USE_ATLAS:
    restore_database.command(
        "collMod",
        "tickets",
        validator=ticket_validator,
        validationLevel="strict",
        validationAction="error",
    )
    print("Recreated index and server-side validator in the restore target.")
else:
    print("Recreated index. Offline path records, but cannot enforce, the server validator.")

print("Restore indexes after repair:", sorted(index["name"] for index in restore_tickets.list_indexes()))

Restore indexes before repair: ['_id_']
Expected: only the automatic _id_ index before manual recreation.
Recreated index. Offline path records, but cannot enforce, the server validator.
Restore indexes after repair: ['_id_', 'status_1_opened_at_-1']


## 6. Test Validation Behavior

Atlas should reject the invalid status after the validator is recreated. The
offline path performs an explicit rule check and labels it as application-level
simulation, not database enforcement.

In [8]:
invalid_document = {
    "ticket_id": 1099,
    "status": "almost_done",
    "priority": "low",
    "subject": "Expected validation failure",
    "opened_at": datetime.now(timezone.utc),
}

if USE_ATLAS:
    try:
        restore_tickets.insert_one(invalid_document)
        raise AssertionError("Atlas accepted a document the recreated validator should reject.")
    except OperationFailure as error:
        print("Expected Atlas validation error code:", error.code)
else:
    allowed_statuses = {"new", "open", "in_progress", "resolved", "closed"}
    simulated_valid = invalid_document["status"] in allowed_statuses
    print("Offline application-level validation result:", simulated_valid)
    assert not simulated_valid

Offline application-level validation result: False


## 7. Compare This Artifact With `mongodump`

| Concern | Canonical Extended JSON exercise | `mongodump` / `mongorestore` |
|---|---|---|
| selected document values | yes | yes |
| BSON type representation | preserved through Extended JSON when parsed correctly | native BSON archive |
| collection options/validator | not recreated automatically | collection metadata/options within documented behavior |
| index definitions | not recreated automatically | included in dump metadata |
| Atlas database users and network rules | no | no, managed separately |
| multi-collection point consistency | not established by this one-collection script | depends on topology, options, and documented tool behavior |

For an Atlas Free database-level backup, use current compatible MongoDB Database
Tools and the documented `mongodump`/`mongorestore` process. This notebook teaches
the recovery evidence sequence and the limitations of a narrower artifact.

## Recovery Record: Complete Before Submission

**Source and separate target:** [record both names]

**Artifact:** [path, bytes, and abbreviated SHA-256]

**Five checks:** [count, identifiers, type, meaningful query, and rule/index check]

**Omissions:** [which metadata and managed configuration did not travel]

**Atlas Free constraint:** [state the current backup limitation and official
alternative]

**Production next step:** [compatible Database Tools, retention, automation,
restore schedule, or broader verification]

**Credential check:** I confirm no URI or password appears in source or output.
[replace with yes]

**License:** prose CC BY-NC-SA 4.0; code MIT; synthetic data CC0.

In [9]:
client.drop_database(SOURCE_DB)
client.drop_database(RESTORE_DB)
client.close()
print("Removed the disposable source and restore databases and closed the client.")

Removed the disposable source and restore databases and closed the client.
